In [1]:
%load_ext autoreload
%autoreload 2

# Library

In [2]:
import os
import torch
import numpy as np
import pandas as pd

from torch.utils.data import DataLoader

# Device

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Current device: {device}")

Current device: cuda


# Load Data

In [52]:
from torch.utils.data import DataLoader

input_size, patch_size = 64, 16

# MMWHS
from datasets.mmwhs_dataset import MMWHS_Dataset

mmwhs_ts = MMWHS_Dataset(root_dir='*/MMWHS/CT/testing_set', data_type='CT')     # TODO: Select from 'CT' data or 'MRI' data

batch_size, num_workers = 4, 1
loader_ts = DataLoader(mmwhs_ts, batch_size=batch_size, shuffle=False,
                        pin_memory=(torch.cuda.is_available()), num_workers=num_workers)

# Load Model

In [63]:
mode = 'PlaneCycle'  # Select from 'PlaneCycle', 'Slice2D' or 'Flatten3D'

# Cfgs dir
MODEL_REPO_PATH = '*/PlaneCycle/models'     # TODO: hubconf root dir
CHECKPOINT_DIR = ...   # TODO: checkpoint folder dir

CHECKPOINT_FILENAME = 'dinov3_vits16_pretrain_lvd1689m-08c60483.pth'
# CHECKPOINT_FILENAME = 'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'
# CHECKPOINT_FILENAME = 'dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'

MODEL_ARCH = 'dinov3_vits16'
# MODEL_ARCH = 'dinov3_vitb16'
# MODEL_ARCH = 'dinov3_vitl16'

FULL_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, CHECKPOINT_FILENAME)

# Load model offline
import sys

sys.path.append(os.path.abspath("*/PlaneCycle"))    # TODO: project root dir
from planecycle.converters.converter import PlaneCycleConverter

# Load model offline
model = torch.hub.load(
    repo_or_dir=MODEL_REPO_PATH,
    model=MODEL_ARCH,
    source='local',
    pretrained=False,
    block_type=mode,
)

# Load pretrained weights
if not os.path.exists(FULL_CHECKPOINT_PATH):
    raise FileNotFoundError(f"[!ERROR!] Fail to find weights path: {FULL_CHECKPOINT_PATH}")

print(f"[*INFO*] Loading pretrained weights: {CHECKPOINT_FILENAME}")
pretrained_weights = torch.load(FULL_CHECKPOINT_PATH, map_location=device)
model.load_state_dict(pretrained_weights, strict=False)

if mode == 'PlaneCycle':
    converter = PlaneCycleConverter(
        cycle_order=('HW', 'DW', 'DH', 'HW'),
        pool_method="PCg"
    )
    model = converter(model)

model.to(device)
model.eval()

[*INFO*] Loading pretrained weights: dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-23): 24 x SelfAttentionBlock(
      (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (attn): SelfAttention(
        (qkv): LinearKMaskedBias(in_features=1024, out_features=3072, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
    )
  )
  (norm)

# Feat DICE

## MMWHS

In [64]:
import math
import torch
import torch.nn.functional as F
from monai.metrics import DiceMetric
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from PIL import Image
import traceback

dice_metric = DiceMetric(
    include_background=False,
    reduction="mean",
)

try:
    scores = {i: [] for i in range(1)}

    with torch.inference_mode():
        for i, (x, y) in enumerate(loader_ts):

            print(f"[*INFO* {i + 1}/{len(loader_ts)}] Inferring...")

            x, y = x.to(device), y.to(device)

            # x: [B, D, C, H, W] -> [B, C, D, H, W]
            x = x.permute(0, 2, 1, 3, 4)

            features = model.get_intermediate_layers(
                x,
                n=1,    # Select last n layers to output
            )

            # y: [B, D, K, H, W] -> [B, K, D, H, W]
            y = y.permute(0, 2, 1, 3, 4)

            for j, feature in enumerate(features):
                # Feature: [B*D, HW, C_feat]
                h_tok = int(math.sqrt(feature.shape[1]))  
                w_tok = h_tok

                if mode == 'PlaneCycle' or mode == 'Slice2D':
                    feature = (
                        feature.permute(0, 2, 1)
                        .contiguous()
                        .view(feature.shape[0], feature.shape[-1], h_tok, w_tok)
                    )
    
                    # Recover to volume: [B, C_feat, D, h, w]
                    feature = (
                        feature.view(y.shape[0], y.shape[2], feature.shape[1], *feature.shape[-2:])
                        .permute(0, 2, 1, 3, 4)
                        .contiguous()
                    )

                if mode == 'Flatten3D':
                    feature = (
                            feature.permute(0, 2, 1)  # [B, C, DHW]
                            .contiguous()
                            .view(feature.shape[0], feature.shape[-1], y.shape[2], input_size // patch_size,
                                  input_size // patch_size)
                        )

                B, C_feat, D, h, w = feature.shape
                d_center = D // 2

                for b in range(B):
                    feat_grid = feature[b]  # [C_feat, D, h, w]

                    K = y.shape[1]  # Number of classes in mask (exclude background if your data does)
                    H, W = y.shape[-2], y.shape[-1]

                    sim_per_class = []

                    # Normalize feat once
                    norm_feat = F.normalize(feat_grid, p=2, dim=0)  # [C_feat, D, h, w]

                    for k in range(K):
                        mask_k = y[b, k]  # [D, H, W]  (one-hot)

                        mask_k_small = F.interpolate(
                            mask_k.unsqueeze(0).unsqueeze(0).float(),  # [1, 1, D, H, W]
                            size=(D, h, w),
                            mode="nearest"
                        ).squeeze(0).squeeze(0)  # [D,h,w]

                        if mask_k_small.sum() < 1:
                            sim_per_class.append(
                                torch.full((D, h, w), -1.0, device=feat_grid.device, dtype=feat_grid.dtype)
                            )
                            continue

                        fg_feat = feat_grid[:, mask_k_small > 0]  # [C_feat, N]
                        proto = fg_feat.mean(dim=1)  # [C_feat]
                        norm_proto = F.normalize(proto, p=2, dim=0)  # [C_feat]

                        sim_map = (norm_feat * norm_proto[:, None, None, None]).sum(dim=0)  # [D, h, w]
                        sim_per_class.append(sim_map)

                    # stack -> [K, D, h, w]
                    sim_stack = torch.stack(sim_per_class, dim=0)

                    # [K, D, H, W]
                    sim_stack = F.interpolate(
                        sim_stack.unsqueeze(0),  # [1, K, D, h, w]
                        size=(D, H, W),
                        mode="trilinear",
                        align_corners=False
                    ).squeeze(0)  # [K, D, H, W]

                    # Argmax
                    pred_label = torch.argmax(sim_stack, dim=0)  # [D, H, W]

                    # GT label map
                    gt_label = torch.argmax(y[b], dim=0)  # [D, H, W]

                    # DiceMetric, pred_oh / gt_oh: [1, K, D, H, W]
                    pred_oh = F.one_hot(pred_label.long(), num_classes=K).permute(3, 0, 1, 2).unsqueeze(0).float()
                    gt_oh = F.one_hot(gt_label.long(), num_classes=K).permute(3, 0, 1, 2).unsqueeze(0).float()

                    dice_metric.reset()
                    dice_metric(pred_oh, gt_oh)
                    dice = dice_metric.aggregate().item()

                    scores[j].append(dice)

    print(scores)

    for k, v in scores.items():
        if len(v) == 0:
            print(f"Layer {k}'s average score: N/A (no valid samples)")
        else:
            print(f"Layer {k}'s average score: {sum(v) / len(v)}")

except torch.cuda.OutOfMemoryError:
    print("[!ERROR!] CUDA out of memory!")
    traceback.print_exc()
except Exception as e:
    print(f"[!ERROR!] {e}")
    traceback.print_exc()

[*INFO* 1/1] Inferring...
{0: [0.3117481768131256, 0.22052356600761414, 0.36496254801750183, 0.2663206458091736]}
Layer 0's average score: 0.2908887341618538
